# Issue 23 — verified mounted-Drive backup and recovery
Default action is a **CPU storage probe using original, already trained JHB Source artifacts**.
No new fitting, inference or Final access occurs in that action. New/resumed city execution
requires a GPU and the unchanged accepted primary recipe. Confirm the previous job state
before starting a replacement. Never rotate accounts to evade quotas.

The implementation is pinned to `4f174bd8971e22e99a57f193eb8e852469762a85`.
Completed runs are archived individually before continuing. A lost in-flight run/epoch is
not certified, and a mounted-filesystem readback is not proof of permanent cloud retention.
Read `docs/reports/issue23-durable-recovery.md`; independent artifact acceptance remains.


In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import hashlib
import json
import shutil
import stat
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

ACTION = "storage-probe"  # storage-probe / new / resume / rescue
CITY = "MEL"  # Accepted separate city; never pool datasets.
RUN_ID = (
    "qyroven-durable-" + CITY.lower() + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
BACKUP_PARENT = Path("/content/drive/MyDrive/Qyroven Issue23 Readiness 20261026")
EXISTING_BACKUP = None  # Existing verified directory for ACTION=resume.
EXISTING_STUDY = None  # Original readable study for ACTION=rescue; never overwrite it.
assert ACTION in {"storage-probe", "new", "resume", "rescue"}
assert CITY in {"AMS", "JHB", "LOA", "MEL", "SPO", "SZH"}

subprocess.run(
    [sys.executable, "-m", "pip", "install", "gdown==5.2.0", "numpy==2.3.3", "ruff==0.17.0"],
    check=True,
)

In [ ]:
import importlib.metadata

import gdown
import numpy as np
import torch

assert torch.__version__.split("+")[0] == "2.11.0", "Review runtime compatibility explicitly"
assert np.__version__ == "2.3.3", (
    "Use a fresh runtime after dependency changes; preserve existing jobs first"
)
if ACTION in {"new", "resume"}:
    assert torch.cuda.is_available(), "Training requires GPU allocation; do not train on CPU"


def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def download(file_id, path, size, checksum):
    if not path.exists():
        gdown.download(id=file_id, output=str(path), quiet=False, use_cookies=False)
    assert path.stat().st_size == size and sha(path) == checksum


def unpack(path, destination):
    destination.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(path) as archive:
        assert archive.testzip() is None
        names = archive.namelist()
        assert len(names) == len(set(names))
        for member in archive.infolist():
            p = Path(member.filename)
            assert not p.is_absolute() and ".." not in p.parts
            assert stat.S_IFMT(member.external_attr >> 16) != stat.S_IFLNK
        archive.extractall(destination)


LOCAL = Path("/content") / RUN_ID
LOCAL.mkdir(exist_ok=False)
with (LOCAL / "runtime-packages.json").open("x") as stream:
    json.dump(
        {
            "python": sys.version,
            "torch": torch.__version__,
            "numpy": np.__version__,
            "packages": {
                d.metadata["Name"]: d.version
                for d in importlib.metadata.distributions()
                if d.metadata["Name"]
            },
        },
        stream,
        indent=2,
    )
bundle = LOCAL / "code.bundle"
download(
    "1MI_ICB2emSMJOtwHm3rnMu8-UxzVUg7z",
    bundle,
    7876741,
    "6acc0cabc1e84e3f1c4c93006aa15e41aea6fecf0fa30eacfb8193aaa494fa91",
)
REPO = LOCAL / "repo"
subprocess.run(["git", "clone", str(bundle), str(REPO)], check=True)
subprocess.run(
    ["git", "checkout", "--detach", "4f174bd8971e22e99a57f193eb8e852469762a85"],
    cwd=REPO,
    check=True,
)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
sys.path.insert(0, str(REPO))
for args in (
    ["unittest", "discover", "-s", "tests", "-q"],
    ["ruff", "check", "."],
    ["ruff", "format", "--check", "."],
):
    subprocess.run([sys.executable, "-m", *args], cwd=REPO, check=True)

In [ ]:
from google.colab import drive

from ev_runner.run import runtime_code
from tools.durable_run_backup import RunBackup, require_drive_mount

drive.mount("/content/drive")
assert BACKUP_PARENT.is_dir(), "Select the existing team artifact folder mapped in DriveFS"
require_drive_mount(BACKUP_PARENT / "mount-check", Path("/content/drive"))
if ACTION == "resume":
    assert EXISTING_BACKUP is not None, "Select the existing verified backup directory"
BACKUP = Path(EXISTING_BACKUP) if ACTION == "resume" else BACKUP_PARENT / (RUN_ID + "-completed")
if ACTION == "storage-probe":
    # These are original JHB models, not newly trained probe models or MEL Sources.
    archive = LOCAL / "original-jhb-sources.zip"
    download(
        "1ebTZQ3lsNdtaR6TLbaxsowYDVxAaJ6ed",
        archive,
        18263480,
        "3ca64e13696ae7aa03ed5686cf2c99e83495eead1580cda4fa27dcdb1c914767",
    )
    unpack(archive, LOCAL / "original-jhb")
    study = LOCAL / "original-jhb/study"
    cohort = json.loads((REPO / "configs/issue23/charged-cohorts-v1/charged-JHB.json").read_text())
    profile = json.loads((REPO / "configs/issue23/source-epoch-study.json").read_text())
    sources = [
        json.loads((study / f"source-seed{s}-complete.json").read_text()) for s in (42, 43, 44)
    ]
    context = {
        "scope": "Original JHB Source-only storage probe; no new fitting/inference/Final",
        "namespace": ["charged", "JHB"],
        "seeds": [42, 43, 44],
        "source_epochs": 10,
        "source_training": profile["source_training"],
        "source_population_sha256": cohort["source_population_sha256"],
        "source_manifest_sha256": cohort["budgets"]["7"]["manifest_sha256"],
        "core_files": runtime_code()["files"],
        "expected_runs": {},
        "environment": sources[0]["run"]["environment"],
    }
    storage = RunBackup(study, BACKUP, context, {})
    for source in sources:
        storage.source(source)
    restored = LOCAL / "restored-source-probe"
    restored.mkdir()
    checked = RunBackup(restored, BACKUP, context, {}, resume=True)
    recovered, targets = checked.restore()
    assert len(recovered) == 3 and not targets
    result = {
        "scope": "Mounted-filesystem Source-only probe; independent cloud readback still required",
        "sources": 3,
        "new_training": False,
        "gpu_training": False,
        "backup_root": str(BACKUP),
        "context_sha256": checked.context_sha,
        "source_sha256": {str(x["seed"]): x["sha256"] for x in recovered},
    }
    with (BACKUP / "probe-result.json").open("x") as stream:
        json.dump(result, stream, indent=2)
    print("DURABLE_SOURCE_STORAGE_PROBE_PASS", json.dumps(result))

In [ ]:
if ACTION in {"new", "resume", "rescue"}:
    # Only the selected city input ZIP is used. Original fingerprints never change.
    catalog = json.loads((REPO / "data/releases/DATA-v1-20261019.json").read_text())
    entries = catalog.get("packages", catalog.get("files", []))
    item = next(
        x for x in entries if x["kind"] == "inputs" and x["name"] == f"inputs-charged-{CITY}-v1.zip"
    )
    archive = LOCAL / item["name"]
    download(item["drive_file_id"], archive, item["size_bytes"], item["sha256"])
    unpack(archive, LOCAL / "inputs")
    INPUTS = LOCAL / "inputs/data/inputs/issue18-input-handoff-v1-20261019"
    dependencies = LOCAL / "primary-dependencies.zip"
    download(
        "13E4lTy6jqCrH6IEQlCDKbcj1Ea-lvKDz",
        dependencies,
        8176124,
        "7f21d35ba89113aed187a644a7b583e70cf83f15f3a87052bc794144ad2076dd",
    )
    unpack(dependencies, LOCAL / "dependencies")
    PRIMARY = LOCAL / "dependencies/primary-analysis.json"
    DENSITY = LOCAL / "dependencies/density-contexts.json"
    assert sha(PRIMARY) == "ecc8b1854128f84ab13070be4ab38d6179d7f15c331f8cf2caf72604576157b6"
    assert sha(DENSITY) == "686c9adf4b56d834787f98dd47d80059e214b7d57192bc4e0778c02a13ef0393"
    COHORT = REPO / f"configs/issue23/charged-cohorts-v1/charged-{CITY}.json"
    OUT = LOCAL / "study"
    common = [
        "--inputs-root",
        str(INPUTS),
        "--cohort",
        str(COHORT),
        "--cohort-sha256",
        sha(COHORT),
        "--primary-analysis",
        str(PRIMARY),
        "--primary-sha256",
        sha(PRIMARY),
        "--backup-root",
        str(BACKUP),
        "--drive-mount",
        "/content/drive",
    ]
    if ACTION == "rescue":
        assert EXISTING_STUDY is not None, "Select the original readable study; do not restart it"
        snapshot = Path(EXISTING_STUDY) / "progress.json"
        if not snapshot.exists():
            snapshot = Path(EXISTING_STUDY) / "matrix.json"
        command = [
            sys.executable,
            "-m",
            "tools.backup_completed_city",
            "--snapshot",
            str(snapshot),
            *common,
        ]
    else:
        command = [
            sys.executable,
            "-m",
            "tools.charged_matched_development",
            "--output",
            str(OUT),
            *common,
        ]
        if ACTION == "resume":
            command.append("--resume-backup")
    print("IDENTIFIED_COMMAND", command)
    with (LOCAL / "execution.log").open("x") as log:
        subprocess.run(command, cwd=REPO, stdout=log, stderr=subprocess.STDOUT, check=True)

In [ ]:
if ACTION in {"new", "resume"}:

    def stage(name, args):
        with (LOCAL / (name + ".log")).open("x") as log:
            subprocess.run(
                [sys.executable, "-m", *map(str, args)],
                cwd=REPO,
                stdout=log,
                stderr=subprocess.STDOUT,
                check=True,
            )
        print("STAGE_PASS", name)

    matrix = OUT / "matrix.json"
    daily = LOCAL / "daily-verification.json"
    states = LOCAL / "latest-states.json"
    analysis = LOCAL / "analysis"
    stage(
        "daily",
        [
            "tools.verify_comparison",
            "--matrix",
            matrix,
            "--inputs-root",
            INPUTS,
            "--output",
            daily,
            "--cache-inputs",
        ],
    )
    stage(
        "states",
        [
            "tools.verify_latest_states",
            "--matrix",
            matrix,
            "--verification",
            daily,
            "--output",
            states,
        ],
    )
    stage(
        "analysis",
        [
            "tools.analyze_charged_matched",
            "--matrix",
            matrix,
            "--verification",
            daily,
            "--cohort",
            COHORT,
            "--cohort-sha256",
            sha(COHORT),
            "--primary-analysis",
            PRIMARY,
            "--primary-sha256",
            sha(PRIMARY),
            "--analysis-profile",
            REPO / "configs/issue23/analysis-profile.json",
            "--inputs-root",
            INPUTS,
            "--density-contexts",
            DENSITY,
            "--density-sha256",
            sha(DENSITY),
            "--output",
            analysis,
        ],
    )
    exports = LOCAL / "exports"
    stage(
        "export",
        [
            "tools.export_development",
            "--matrix",
            matrix,
            "--verification",
            daily,
            "--latest-states",
            states,
            "--source-root",
            OUT,
            "--output",
            exports,
            "--attachment",
            "city-analysis.json=" + str(analysis / "summary.json"),
            "--attachment",
            "city-runs.csv=" + str(analysis / "runs.csv"),
        ],
    )
    # Canonical complete handoff is separate from incremental recovery receipts.
    handoff = BACKUP_PARENT / (RUN_ID + "-export")
    handoff.mkdir(exist_ok=False)
    catalog = json.loads((exports / "catalog.json").read_text())
    for part in catalog["parts"]:
        source = exports / part["name"]
        pending = handoff / (part["name"] + ".pending")
        with source.open("rb") as src, pending.open("xb") as dst:
            shutil.copyfileobj(src, dst)
            dst.flush()
            os.fsync(dst.fileno())
        assert pending.stat().st_size == part["bytes"] and sha(pending) == part["sha256"]
        pending.rename(handoff / part["name"])
    with (handoff / "catalog.json").open("xb") as dst:
        dst.write((exports / "catalog.json").read_bytes())
    assert sha(handoff / "catalog.json") == sha(exports / "catalog.json")
    metadata = handoff / "invocation-metadata.zip"
    with zipfile.ZipFile(metadata, "x", compression=zipfile.ZIP_DEFLATED) as z:
        for path in [
            *LOCAL.glob("*.log"),
            LOCAL / "runtime-packages.json",
            PRIMARY,
            DENSITY,
            COHORT,
            REPO / "configs/issue23/analysis-profile.json",
        ]:
            z.write(path, path.name)
    print(
        "COMPLETE_MOUNTED_HANDOFF",
        str(handoff),
        "Cloud visibility/download/restore and peer acceptance still required",
    )
    # Stop here. No Final lock/evaluation, automatic merge or issue closure.